# Week 3: Generate Model-Driven Training Data

## Purpose

This notebook queries CodeLlama to get the **actual top-K tokens** it would predict for each prompt, then saves them to CSV for manual labeling.

## Process

1. Load model
2. For each base prompt, get top-20 most probable next tokens
3. Save: `prompt`, `next_token`, `probability`, `full_text` to CSV
4. **STOP** - Manual labeling needed

## Output

`training_data_unlabeled.csv` with columns:
- `prompt`: The base prompt
- `next_token`: The actual token model predicts
- `probability`: Token probability
- `full_text`: prompt + next_token
- `label`: EMPTY (to be filled manually)

---

In [ ]:
# Cell 1: Install
!pip install -q transformers torch accelerate pandas

In [ ]:
# Cell 2: Imports
import torch
import numpy as np
import pandas as pd
from typing import List, Dict
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
print("✅ Imports complete")

In [ ]:
# Cell 3: Load Model
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.eval()

print(f"✅ Model loaded on {model.device}")

In [ ]:
# Cell 4: Define base prompts

# These are prompts where we EXPECT the model to be uncertain about CODE-specific tokens
CODE_PROMPTS = [
    'The authentication is done using',
    'Password hashing uses',
    'Tokens are generated with',
    'The database is',
    'The frontend uses',
    'State management is',
    'We import',
    'The ORM is',
    'The API is built with',
    'GraphQL is implemented with',
    'The model architecture is',
    'Training uses optimizer =',
    'Deployment is on',
    'CI/CD is',
    'Unit tests use',
    'The bundler is',
    'Logging is done with',
    'Message queue is',
    'Input validation uses',
    'HTTP requests use',
]

# These are prompts where we EXPECT the model to be uncertain about LANGUAGE tokens
LANGUAGE_PROMPTS = [
    'The authentication is',
    'Verification is done',
    'The user id is',
    'Processing is handled',
    'Data is stored',
    'Tokens are validated',
    'The session is created',
    'The code works by',
    'The function is responsible for',
    'This approach',
    'The main purpose is to',
    'To implement authentication,',
    'First, you need to',
    'Make sure to',
    'Unlike other methods,',
    'The difference is',
    'How does the authentication',
    'What are the benefits of',
    'When should you',
    'The authentication system',
]

ALL_PROMPTS = [
    {'prompt': p, 'expected_type': 'code'} for p in CODE_PROMPTS
] + [
    {'prompt': p, 'expected_type': 'language'} for p in LANGUAGE_PROMPTS
]

print(f"Total prompts: {len(ALL_PROMPTS)}")
print(f"  CODE prompts: {len(CODE_PROMPTS)}")
print(f"  LANGUAGE prompts: {len(LANGUAGE_PROMPTS)}")

In [ ]:
# Cell 5: Query model for top-K predictions

def softmax(logits: np.ndarray) -> np.ndarray:
    logits_stable = logits - np.max(logits)
    exp_logits = np.exp(logits_stable)
    return exp_logits / np.sum(exp_logits)

def get_top_k_predictions(prompt: str, top_k: int = 20) -> List[Dict]:
    """
    Get top-K most probable next tokens for a given prompt.
    Returns list of {token, probability, full_text}
    """
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits[0, -1, :].cpu().numpy()
    
    probs = softmax(logits)
    top_indices = np.argsort(probs)[-top_k:][::-1]
    
    results = []
    for idx in top_indices:
        token = tokenizer.decode([idx])
        prob = float(probs[idx])
        full_text = prompt + token
        
        results.append({
            'token': token,
            'probability': prob,
            'full_text': full_text
        })
    
    return results

print("✅ Query function ready")

In [ ]:
# Cell 6: Generate dataset

TOP_K = 20  # Get top-20 most probable tokens for each prompt

print(f"Querying model for top-{TOP_K} predictions...")
print(f"This will generate ~{len(ALL_PROMPTS) * TOP_K} training examples\n")

data_rows = []

for prompt_data in tqdm(ALL_PROMPTS, desc="Processing prompts"):
    prompt = prompt_data['prompt']
    expected_type = prompt_data['expected_type']
    
    # Get top-K predictions from model
    predictions = get_top_k_predictions(prompt, top_k=TOP_K)
    
    # Create a row for each prediction
    for pred in predictions:
        data_rows.append({
            'prompt': prompt,
            'expected_type': expected_type,  # Hint for labeling
            'next_token': pred['token'],
            'probability': pred['probability'],
            'full_text': pred['full_text'],
            'label': ''  # TO BE FILLED MANUALLY
        })

# Create DataFrame
df = pd.DataFrame(data_rows)

print(f"\n✅ Generated {len(df)} training examples")
print(f"\nDataset preview:")
print(df.head(10))

In [ ]:
# Cell 7: Save to CSV

OUTPUT_FILE = 'training_data_unlabeled.csv'

df.to_csv(OUTPUT_FILE, index=False)

print(f"\n" + "="*80)
print(f"DATASET SAVED: {OUTPUT_FILE}")
print(f"="*80)
print(f"\nTotal examples: {len(df)}")
print(f"\nColumns:")
for col in df.columns:
    print(f"  - {col}")

print(f"\n" + "="*80)
print(f"NEXT STEPS:")
print(f"="*80)
print(f"\n1. Open {OUTPUT_FILE}")
print(f"2. Fill the 'label' column with:")
print(f"   - 'code' if next_token represents code-specific uncertainty")
print(f"   - 'language' if next_token represents word choice uncertainty")
print(f"\n3. Guidelines for labeling:")
print(f"   CODE examples:")
print(f"     - 'JWT', 'OAuth', 'Firebase' (specific tech names)")
print(f"     - 'Py', 'Node', 'Post' (start of tech words)")
print(f"     - Any token that indicates specific library/framework/tool")
print(f"\n   LANGUAGE examples:")
print(f"     - 'a', 'the', 'an' (articles)")
print(f"     - 'done', 'handled', 'implemented' (generic verbs)")
print(f"     - Any token that is generic English")
print(f"\n4. Save as 'training_data_labeled.csv'")
print(f"5. Run the training notebook with the labeled data")
print(f"\n" + "="*80)

In [ ]:
# Cell 8: Show examples to help with labeling

print("\n" + "="*80)
print("EXAMPLE ENTRIES FOR LABELING")
print("="*80)

# Show some examples from CODE prompts
print(f"\nExamples from 'The authentication is done using':")
sample = df[df['prompt'] == 'The authentication is done using'].head(10)
for _, row in sample.iterrows():
    print(f"  Token: '{row['next_token']}'  (prob={row['probability']:.3f})")
    print(f"    Full: '{row['full_text']}'")
    print(f"    Label: ??? (you decide: 'code' or 'language')\n")

# Show some examples from LANGUAGE prompts  
print(f"\nExamples from 'The authentication is':")
sample = df[df['prompt'] == 'The authentication is'].head(10)
for _, row in sample.iterrows():
    print(f"  Token: '{row['next_token']}'  (prob={row['probability']:.3f})")
    print(f"    Full: '{row['full_text']}'")
    print(f"    Label: ??? (you decide: 'code' or 'language')\n")

print("="*80)